### Imports

In [39]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import plotly.express as px
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

In [40]:
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[1]

file_path = PROJECT_ROOT / "data" / "processed" / "calldata_20251019_processed_v4.csv"

OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

FIG_DIR = PROJECT_ROOT / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Anomaly Detection

### Data set up

In [41]:
spark = SparkSession.builder.appName("Seattle911").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df_spark = spark.read.csv(str(file_path), header=True, inferSchema=True)

# print(df_spark.columns)
# print(df_spark.count())

In [42]:
df_spark = (
    df_spark.filter(
          (F.col("dispatch_neighborhood").isNotNull()) &
          (F.col("dispatch_sector").isNotNull()) &
          (F.col("dispatch_latitude").isNotNull()) &
          (F.col("dispatch_longitude").isNotNull()) &
          (~F.col("dispatch_neighborhood").like("%REDACTED%")) &
          (~F.col("dispatch_sector").like("%REDACTED%")) &
          (~F.col("dispatch_latitude").like("%REDACTED%")) &
          (~F.col("dispatch_longitude").like("%REDACTED%"))
      )
)

In [43]:
df_spark = (df_spark
            .withColumn("event_time", F.to_timestamp("cad_event_original_time_queued", "MM/dd/yyyy hh:mm:ss a"))
            .withColumn("date", F.to_date("event_time"))
            .withColumn("hour", F.hour("event_time")))

#df_spark.select("cad_event_original_time_queued", "event_time", "date", "hour", "call_type", "call_sign_total_service_time_s")\
#        .show(10, truncate=False)

In [44]:
hourly_calls = (df_spark.groupBy("date", "hour", "call_type").agg( F.count("*").alias("total_calls"), 
                        F.avg("call_sign_total_service_time_s").alias("avg_service_time")).orderBy("date", "hour"))
#hourly_calls.show(10)

### Response Time Anomalies by Call Type + Time of Day

This identifies calls whose response times are unusually long or short compared to what would normally be expected for that specific call type, time of day, and dispatch area.

In [45]:
hourly = hourly_calls.toPandas().copy()

hourly["datetime"] = (pd.to_datetime(hourly["date"]) + pd.to_timedelta(hourly["hour"], unit="h"))
hourly["hour_num"] = hourly["hour"].astype(int)
le = LabelEncoder()
hourly["call_type_encoded"] = le.fit_transform(hourly["call_type"])

X = hourly[["avg_service_time", "total_calls", "hour_num", "call_type_encoded"]].values


contamination_values = [0.01, 0.02, 0.05]
print("Contamination sweep for delayed-response detection:")
for c in contamination_values:
    model = IsolationForest(
        n_estimators=200,
        max_samples="auto",
        contamination=c,
        random_state=42,
        n_jobs=-1,
    )
    preds = model.fit_predict(X)
    flags = (preds == -1).astype(int)
    frac_anom = flags.mean()
    print(f"contamination={c:.3f} → {frac_anom*100:.2f}% of hours flagged")

best_contamination = 0.02

iso_model = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination=best_contamination,
    random_state=42,
    n_jobs=-1,
)

preds = iso_model.fit_predict(X)
hourly["iso_anomaly_flag"] = (preds == -1).astype(int)
hourly["iso_score"] = iso_model.decision_function(X)
hourly["iso_label"] = hourly["iso_anomaly_flag"].map(
    {1: "Delayed Response", 0: "Normal"}
)


##########################
# Please uncomment to plot
#########################


# fig = px.scatter(
#     hourly,
#     x="datetime",
#     y="avg_service_time",
#     color="call_type",                    
#     symbol="iso_label",                   
#     symbol_map={"Normal": "circle", "Delayed Response": "diamond"},
#     title="Incident Flow Timeline – Slow Response Time Anomalies (Isolation Forest)",
#     labels={
#         "datetime": "Date / Hour",
#         "avg_service_time": "Average Response Time (sec)",
#         "call_type": "Call Type",
#         "iso_label": "Status",
#     },
#     hover_data={
#         "datetime": True,
#         "avg_service_time": True,
#         "total_calls": True,
#         "call_type": True,
#         "iso_label": True,
#         "hour": True,
#         "date": True,
#     },
# )

# # Fade normal points
# fig.for_each_trace(
#     lambda t: t.update(
#         marker=dict(size=6, opacity=0.25)
#     ) if "Normal" in t.name else None
# )

# fig.for_each_trace(
#     lambda t: t.update(
#         marker=dict(
#             size=7,                
#             opacity=0.95,
#             line=dict(width=1.2, color="red"),
#         )
#     ) if "Delayed Response" in t.name else None
# )

# fig.update_layout(
#     height=600,
#     width=1200,
#     legend_title_text="Call Type / Status",
# )

# fig.show()

# # Save figure

# fig.write_image(FIG_DIR / "incident_flow_timeline_response_time_anomalies.png", scale=3)

Contamination sweep for delayed-response detection:
contamination=0.010 → 1.00% of hours flagged
contamination=0.020 → 2.00% of hours flagged
contamination=0.050 → 5.00% of hours flagged


In [46]:
#hourly_calls_w_cad = (df_spark.select("cad_event_number", "date", "hour", "call_type", "call_sign_total_service_time_s")
#                      .join(hourly_calls, on=["date", "hour", "call_type"], how="left"))
#hourly_calls_w_cad.select("cad_event_number", "date", "hour", "call_type","total_calls", "avg_service_time", "is_anomaly").show(10, truncate=False)

In [47]:
iso_pd = hourly[["date", "hour", "call_type", "iso_anomaly_flag", "iso_score"]]
iso_spark = spark.createDataFrame(iso_pd)

hourly_calls_iso = hourly_calls.join(iso_spark, on=["date", "hour", "call_type"], how="left")

hourly_calls_w_cad = (df_spark.select("cad_event_number","date","hour","call_type","call_sign_total_service_time_s")\
                      .join(hourly_calls_iso, on=["date", "hour", "call_type"], how="left"))

hourly_calls_w_cad.show(10, truncate=False)

+----------+----+-----------------------------+----------------+------------------------------+-----------+------------------+----------------+-------------------+
|date      |hour|call_type                    |cad_event_number|call_sign_total_service_time_s|total_calls|avg_service_time  |iso_anomaly_flag|iso_score          |
+----------+----+-----------------------------+----------------+------------------------------+-----------+------------------+----------------+-------------------+
|2023-11-01|3   |911                          |2023000316019   |31                            |12         |1744.8333333333333|0               |0.11425422725122175|
|2023-11-01|3   |911                          |2023000316020   |345                           |12         |1744.8333333333333|0               |0.11425422725122175|
|2023-11-01|3   |911                          |2023000316021   |809                           |12         |1744.8333333333333|0               |0.11425422725122175|
|2023-11-01|3   

This scatterplot shows the hourly volume of 911 calls over time, with each point representing the number of calls received in a single hour. The x-axis moves chronologically from late 2023 to 2025, while the y-axis shows how many calls occurred in that hour. Each dot is colored by the type of incident (for example, 911 calls, officer-initiated events, on-view incidents, etc.), which helps illustrate how different categories contribute to the total workload.

The chart also marks anomalies—hours where the number of calls was unusually high or unusually low based on statistical patterns for that call type. These anomaly points appear as “×” markers, making them stand out from the normal circular points. Clusters of “×” symbols indicate periods where call volume deviated strongly from normal behavior, highlighting spikes or rare surges in demand.

Because this figure is interactive, you can click on the legend entries to show or hide specific call types or anomaly categories, making it easy to isolate patterns, reduce clutter, or focus on particular incident types.

Overall, the visualization helps you see not only the day-to-day rhythm of 911 activity but also the outliers that may correspond to unusual events, emergencies, or data irregularities.

### Call Type Mismatch Anomalies (Initial vs Final)

Find cases where the initial classification differs significantly from the final (e.g., “Aid Response” → “Structure Fire”).

Build a confusion matrix or frequency table of transitions; use rare transitions as anomalies.

In [48]:
INITIAL_COL = "initial_call_type_mapping"
FINAL_COL = "final_call_type_mapping"
#df_spark.select(INITIAL_COL, FINAL_COL).show(5)

In [49]:
transition_counts = (df_spark.groupBy(INITIAL_COL, FINAL_COL).agg(F.count("*").alias("count")).orderBy(F.desc("count")))
#transition_counts.show(20, truncate=False)

In [50]:
total_calls = df_spark.count()
transition_counts = (transition_counts
    .withColumn("pct", F.col("count") / F.lit(total_calls))
    .withColumn("is_anomaly", (F.col("pct") < 0.005).cast("int")))

#transition_counts.orderBy("pct").show(20, truncate=False)

In [51]:
transition_counts = transition_counts.withColumn("is_self_transition", (F.col(INITIAL_COL) == F.col(FINAL_COL)).cast("int"))

In [52]:
transition_pd = transition_counts.toPandas()
if "is_self_transition" in transition_pd.columns:
    transition_pd = transition_pd[transition_pd["is_self_transition"] == 0]
transition_pd["transition_label"] = transition_pd[INITIAL_COL] + " → " + transition_pd[FINAL_COL]
top_transitions = transition_pd.sort_values("pct", ascending=False).head(15)

##########################
# Please uncomment to plot
#########################

# plt.figure(figsize=(10, 6))
# sns.barplot(data=top_transitions, x="pct", y="transition_label", hue="is_anomaly", dodge=False, palette={0: "gray", 1: "red"})
# plt.title("Top Call Type Transitions — Rare Mismatches Highlighted", fontsize=14)
# plt.xlabel("Proportion of Calls")
# plt.ylabel("Initial → Final Call Type")
# plt.legend(title="Anomaly", bbox_to_anchor=(1.05, 1), loc="upper left")
# plt.tight_layout()
# plt.show()

This bar chart shows the most common transitions from an initial call type to a final call type, with the length of each bar representing how frequently that transition occurs and the color indicating whether the transition is considered an anomaly.

In [53]:
##########################
# Please uncomment to plot
#########################


# rare_transitions = transition_pd[transition_pd["is_anomaly"] == 1]
# sns.barplot(data=rare_transitions.sort_values("pct", ascending=False).head(15),x="pct", y="transition_label", color="red")
# # plt.savefig("hourly_911_call_typeanomalies.png", dpi=300, bbox_inches="tight")
# plt.title("Rare Call Type Mismatches (<0.5% Frequency)")

This chart highlights the rarest call-type transitions in the data, with each bar showing a transition that occurs in less than 0.5% of calls and the bar length indicating just how uncommon that mismatch is.

In [54]:
CAD_COL = "cad_event_number"
transition_w_cad = (df_spark.select(CAD_COL, INITIAL_COL, FINAL_COL).join(transition_counts, on=[INITIAL_COL, FINAL_COL], how="left"))
#transition_w_cad.select(CAD_COL, INITIAL_COL, FINAL_COL, "count", "pct", "is_anomaly").show(10, truncate=False)

### Geographic Outlier Detection (Spatial Anomalies)

This metric checks whether a call’s location is unusually far from where similar incidents typically occur, helping identify mis-geocoded or unexpectedly placed calls whose coordinates don’t match the normal patterns for that call type or area.

In [55]:
LAT_COL = "dispatch_latitude"
LON_COL = "dispatch_longitude"
GROUP_COL = "dispatch_sector"

geo_stats = (df_spark.groupBy(GROUP_COL).agg(
        F.avg(LAT_COL).alias("mean_lat"),
        F.avg(LON_COL).alias("mean_lon"),
        F.stddev(LAT_COL).alias("std_lat"),
        F.stddev(LON_COL).alias("std_lon"),
        F.count("*").alias("n_calls")))

df_geo = df_spark.join(geo_stats, on=GROUP_COL, how="left")
df_geo = (df_geo
    .withColumn("z_lat", (F.col(LAT_COL) - F.col("mean_lat")) / (F.col("std_lat") + F.lit(1e-6)))
    .withColumn("z_lon", (F.col(LON_COL) - F.col("mean_lon")) / (F.col("std_lon") + F.lit(1e-6)))
    .withColumn("spatial_score", F.sqrt(F.col("z_lat")**2 + F.col("z_lon")**2))
    .withColumn("is_geo_anomaly", (F.col("spatial_score") > 3).cast("int")))

#df_geo.select(GROUP_COL, LAT_COL, LON_COL, "spatial_score", "is_geo_anomaly").show(10, truncate=False)

In [56]:
##########################
# Please uncomment to plot
#########################


# geo_pd = (df_geo.select(GROUP_COL, LAT_COL, LON_COL, "is_geo_anomaly")
#                 .dropna(subset=[LAT_COL, LON_COL])  
#                 .sample(fraction=0.1, seed=42)     
#                 .limit(10000)                       
#                 .toPandas())

# plt.figure(figsize=(8, 6))
# sns.scatterplot(data=geo_pd,
#                x=LON_COL, y=LAT_COL,
#                hue="is_geo_anomaly",
#                palette={0: "gray", 1: "red"},
#                alpha=0.7)
# plt.title("Geographic Outlier Detection — Red = Spatial Anomaly", fontsize=14)
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.legend(title="Anomaly")
# plt.tight_layout()
# # plt.savefig("hourly_911_geographic_anomalies.png", dpi=300, bbox_inches="tight")
# plt.xticks([])
# plt.yticks([])
# plt.show()

This scatterplot shows the geographic distribution of calls, where most points cluster along typical latitude–longitude patterns, and the red points highlight calls whose locations are statistically unusual or inconsistent with where similar incidents normally occur.

### Temporal Burst Detection

This method identifies sudden spikes in call volume within a neighborhood or beat by applying moving-window detect abnormal increases in calls over short time intervals.

In [57]:
df_spark = df_spark.withColumn("event_hour", F.date_trunc("hour", F.col("event_time")))
hourly_counts = (df_spark.groupBy("event_hour", "call_type").agg(F.count("*").alias("total_calls")).orderBy("event_hour"))
#hourly_counts.show(10, truncate=False)

In [58]:
w = Window.orderBy("event_hour").rowsBetween(-6, 6)
hourly_counts = (hourly_counts
            .withColumn("rolling_mean", F.avg("total_calls").over(w))
            .withColumn("rolling_std", F.stddev("total_calls").over(w))
            .withColumn("z_score", (F.col("total_calls") - F.col("rolling_mean")) / (F.col("rolling_std") + F.lit(1e-6)))
            .withColumn("is_burst", (F.col("z_score") > 2.5).cast("int")))
#hourly_counts.select("event_hour", "total_calls", "rolling_mean", "z_score", "is_burst").show(10, truncate=False)

In [59]:
hourly_pd = hourly_counts.toPandas()
#plt.figure(figsize=(14,6))
#sns.lineplot(data=hourly_pd, x="event_hour", y="total_calls", label="Total Calls", color="gray")
#sns.scatterplot(data=hourly_pd[hourly_pd["is_burst"] == 1], x="event_hour", y="total_calls", color="red", label="Burst Anomaly", s=60)
#plt.title("Temporal Burst Detection — Sudden Increases in 911 Call Volume", fontsize=14)
#plt.xlabel("Time")
#plt.ylabel("Number of Calls per Hour")
#plt.legend()
#plt.tight_layout()
#plt.savefig("hourly_911_temporal_burst_anomalies.png", dpi=300, bbox_inches="tight")
#plt.show()

This chart shows the hourly 911 call volume over time, with the gray line representing normal fluctuations and the red points highlighting hours where call volume spiked unusually high compared to surrounding periods, indicating potential burst anomalies.

In [60]:
sector_window = Window.partitionBy("dispatch_sector").orderBy("event_hour").rowsBetween(-6, 6)
sector_bursts = (df_spark
    .withColumn("event_hour", F.date_trunc("hour", F.col("event_time")))
    .groupBy("dispatch_sector", "event_hour")
    .agg(F.count("*").alias("total_calls"))
    .withColumn("rolling_mean", F.avg("total_calls").over(sector_window))
    .withColumn("rolling_std", F.stddev("total_calls").over(sector_window))
    .withColumn("z_score", (F.col("total_calls") - F.col("rolling_mean")) / (F.col("rolling_std") + F.lit(1e-6)))
    .withColumn("is_burst", (F.col("z_score") > 2.5).cast("int")))

In [61]:
##########################
# Please uncomment to plot
#########################


# sector_bursts_pd = sector_bursts.toPandas()

# plt.figure(figsize=(14,6))
# sns.lineplot(data=sector_bursts_pd, x="event_hour", y="total_calls", label="Total Calls", color="gray")
# sns.scatterplot(data=sector_bursts_pd[sector_bursts_pd["is_burst"] == 1],x="event_hour", y="total_calls",\
#                color="red", label="Burst Anomaly", s=60)
# plt.title("Temporal Burst Detection — Sudden Increases in 911 Call Volume per Sector", fontsize=14)
# plt.xlabel("Time")
# plt.ylabel("Number of Calls per Hour")
# plt.legend()
# plt.tight_layout()
# # plt.savefig("hourly_911_temporal_burst_anomalies_2.png", dpi=300, bbox_inches="tight")
# plt.show()

This chart shows hourly 911 call volume within a specific sector, with the gray line representing normal call levels and the red points marking hours where that sector experienced an unusually sharp spike in calls compared to its typical hourly patterns.

In [62]:
df_w_cad_hourly = (df_spark.select("cad_event_number", "event_hour") .join(hourly_counts, on="event_hour", how="left"))
#df_w_cad_hourly.select("cad_event_number","event_hour","total_calls","rolling_mean","z_score","is_burst").show(10, truncate=False)

df_w_cad_sector = (df_spark.select("cad_event_number", "event_hour", "dispatch_sector").join(sector_bursts, on=["event_hour", "dispatch_sector"], how="left"))
#df_w_cad_sector.select("cad_event_number","dispatch_sector","event_hour","total_calls","z_score","is_burst").show(10, truncate=False)

### Dispatch Routing Anomalies

This metric checks whether a call was dispatched to the appropriate station or sector based on its location, helping identify mismatches or inefficient routing where a unit was sent from farther away than expected. Unlike the geographic outlier metric—which detects whether the call itself is placed in an unusual location—this one evaluates whether the response assignment makes sense geographically.

In [63]:
def haversine_distance(lat1, lon1, lat2, lon2):
    return (F.lit(6371) * 2 * F.asin(
            F.sqrt(F.pow(F.sin((F.radians(lat2 - lat1)) / 2), 2) + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2))\
                   * F.pow(F.sin((F.radians(lon2 - lon1)) / 2), 2))))

In [64]:
sector_centroids = (df_spark.groupBy("dispatch_sector")
    .agg(F.avg("dispatch_latitude").alias("sector_lat"), F.avg("dispatch_longitude").alias("sector_lon"), F.count("*").alias("n_calls")))

In [65]:
df_dist = df_spark.join(sector_centroids, on="dispatch_sector", how="left")
df_dist = df_dist.withColumn("distance_km",
    haversine_distance( F.col("dispatch_latitude"), F.col("dispatch_longitude"), F.col("sector_lat"), F.col("sector_lon")))

In [66]:
w = Window.partitionBy("dispatch_sector")
df_dist = (df_dist
        .withColumn("mean_dist", F.avg("distance_km").over(w))
        .withColumn("std_dist", F.stddev("distance_km").over(w))
        .withColumn("z_score", (F.col("distance_km") - F.col("mean_dist")) / (F.col("std_dist") + F.lit(1e-6)))
        .withColumn("is_routing_anomaly", (F.col("z_score") > 3).cast("int")))

#df_dist.select("dispatch_sector", "dispatch_latitude", "dispatch_longitude", "distance_km", "z_score", "is_routing_anomaly").show(10, truncate=False)

In [67]:
##########################
# Please uncomment to plot
#########################



# routing_pd = (df_dist.select("dispatch_latitude", "dispatch_longitude", "is_routing_anomaly").dropna(subset=["dispatch_latitude", "dispatch_longitude"])
#                      .sample(fraction=0.05, seed=42).toPandas())

# plt.figure(figsize=(8, 6))
# sns.scatterplot(data=routing_pd, x="dispatch_longitude", y="dispatch_latitude", hue="is_routing_anomaly", 
#                palette={0: "gray", 1: "red"}, alpha=0.7)
# plt.title("Dispatch Routing Anomalies — Red = Calls Far from Assigned Sector", fontsize=14)
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.legend(title="Anomaly")
# plt.tight_layout()
# plt.xticks([])
# plt.yticks([])
# plt.show()

This scatterplot maps where calls were dispatched, with each point showing the call’s latitude and longitude. Most points are gray, indicating normal routing where the assigned sector is geographically appropriate. The red points highlight potential routing anomalies which are calls that appear to have been assigned to a sector unusually far from their actual location.

### Summarize / Putting all results together

In [68]:
print("Response Time Anomalies DF")
print(hourly_calls_w_cad.count())
#hourly_calls_w_cad.show(5)
print("Call Type Mismatch Anomalies DF")
print(transition_w_cad.count())
#transition_w_cad.show(5)
print("Georgraphic Outlier Detection DF")
print(df_geo.count())
#df_geo.show(5)
print("Temporal Burst Detection DF")
print(df_w_cad_hourly.count())
#df_w_cad_hourly.show(5)
print("per sector")
print(df_w_cad_sector.count())
#df_w_cad_sector.show(5)
print("Dispatch Routing Anomalies")
print(df_dist.count())
#df_dist.show(5)

Response Time Anomalies DF


463625
Call Type Mismatch Anomalies DF
463625
Georgraphic Outlier Detection DF
463625
Temporal Burst Detection DF


1603053
per sector
463625
Dispatch Routing Anomalies
463625


In [69]:
df_response = hourly_calls_w_cad.select("cad_event_number", "iso_anomaly_flag")
df_geo_flag = df_geo.select("cad_event_number", "is_geo_anomaly")
df_routing = df_dist.select("cad_event_number", "is_routing_anomaly")
df_burst = df_w_cad_hourly.select("cad_event_number", "is_burst") 
df_type = transition_w_cad.select("cad_event_number", "is_anomaly").withColumnRenamed("is_anomaly", "is_type_anomaly")

df_response = df_response.withColumnRenamed("iso_anomaly_flag", "is_response_anomaly")
df_burst = df_burst.withColumnRenamed("is_burst", "is_burst_anomaly")

df_final_anomaly_detection = (
    df_spark
    .select("cad_event_number", "dispatch_sector", "call_type", "event_time")
    .join(df_response, on="cad_event_number", how="left")
    .join(df_burst, on="cad_event_number", how="left")
    .join(df_type, on="cad_event_number", how="left")
    .join(df_geo_flag, on="cad_event_number", how="left")
    .join(df_routing, on="cad_event_number", how="left")
)

df_final_anomaly_detection = df_final_anomaly_detection.withColumn(
    "is_any_anomaly",
    (F.col("is_response_anomaly") +
    F.col("is_burst_anomaly") +
    F.col("is_type_anomaly") +
    F.col("is_geo_anomaly") +
    F.col("is_routing_anomaly")
    ).cast("int"))


#df_final_anomaly_detection.select("cad_event_number","is_response_anomaly","is_burst_anomaly","is_type_anomaly",
#    "is_geo_anomaly","is_routing_anomaly","is_any_anomaly").show(10, truncate=False)

is_response_anomaly → Flags calls that occurred during hours where response times or call volumes were unusually high or low compared to typical patterns for that call type.

is_burst_anomaly → Marks time periods where there was a sudden short-term spike in total 911 call volume — a temporal “burst” indicating potential system overload or major incidents.

is_type_anomaly → Indicates calls where the initial classification of the incident didn’t match the final call type, highlighting possible mislabeling or reclassification during dispatch.

is_geo_anomaly → Flags calls whose location coordinates are unusually far from where similar calls or that dispatch sector typically occur — potential geographic outliers or coordinate errors.

is_routing_anomaly → Identifies calls that were dispatched to a sector or unit far from the incident’s actual location, suggesting routing mismatches or boundary assignment issues.

is_any_anomaly → A combined indicator showing whether a call was flagged by any of the five anomaly detection methods above (1 = at least one anomaly detected).

### Creating Tables for Visualization

In [70]:
spark_meta = df_spark.select("cad_event_number","dispatch_sector","dispatch_neighborhood").dropDuplicates(["cad_event_number"])
anomaly_table = (df_w_cad_hourly
        .select("cad_event_number", "event_hour", "call_type",
                "total_calls", "rolling_mean", "rolling_std", "is_burst")
        .withColumn("date", F.to_date("event_hour"))
        .withColumn("hour", F.hour("event_hour"))
        .withColumnRenamed("is_burst", "is_anomaly")
        .withColumnRenamed("call_type", "call_type_filtered")
        .withColumnRenamed("rolling_mean", "mean_calls")
        .withColumnRenamed("rolling_std", "std_calls"))


anomaly_table = (anomaly_table.join(spark_meta, on="cad_event_number", how="left")
        .select("cad_event_number","is_anomaly","date","hour","call_type_filtered","total_calls",
            "mean_calls","std_calls","dispatch_sector","dispatch_neighborhood"))

anomaly_table = anomaly_table.filter((F.col("dispatch_neighborhood").isNotNull()) & (~F.col("dispatch_neighborhood").isin("-", "Unknown")))

#print(anomaly_table.count())
#print(anomaly_table.columns)
csv_path = OUTPUT_DIR / "burst_anomaly_table.csv"

anomaly_table = anomaly_table.toPandas()
anomaly_table.to_csv(csv_path, index=False)

In [71]:
spark_meta = (df_spark.select("cad_event_number", "dispatch_sector", "dispatch_neighborhood")
              .dropDuplicates(["cad_event_number"]))

anomaly_table = (hourly_calls_w_cad
        .select("cad_event_number","date","hour","call_type","total_calls","iso_anomaly_flag","iso_score")
        .withColumnRenamed("iso_anomaly_flag", "is_anomaly")
        .withColumnRenamed("call_type", "call_type_filtered"))

anomaly_table = (anomaly_table
        .join(spark_meta, on="cad_event_number", how="left")
        .select("cad_event_number","is_anomaly","date","hour", "call_type_filtered","total_calls","iso_score","dispatch_sector",
            "dispatch_neighborhood"))

#anomaly_table = anomaly_table.filter((F.col("dispatch_neighborhood").isNotNull())&(~F.col("dispatch_neighborhood").isin("-", "Unknown")))

#print(anomaly_table.count())
#print(anomaly_table.columns)

csv_path = OUTPUT_DIR / "response_anomaly_table.csv"

anomaly_table_pd = anomaly_table.toPandas()
anomaly_table_pd.to_csv(csv_path, index=False)

## Validating

**Valdiating Response Time Anomalies**

In [72]:
##########################
# Please uncomment to plot
#########################


# total_anomalies = hourly_calls_iso.filter(F.col("iso_anomaly_flag") == 1).count()
# total_rows = hourly_calls_iso.count()

# percent_anomalies = (total_anomalies / total_rows) * 100
# print(f"{total_anomalies} anomalies detected ({percent_anomalies:.2f}% of all hourly records)")


# means = (hourly_calls_iso.groupBy("iso_anomaly_flag").agg(F.mean("avg_service_time").alias("mean_service_time")).toPandas())
# means["iso_anomaly_flag"] = means["iso_anomaly_flag"].map({0: "Normal", 1: "Anomaly"})
# print("\nMean average service time by Isolation Forest anomaly flag:")
# print(means)

# hourly_pd = hourly_calls_iso.toPandas()

# hourly_pd["datetime"] = (pd.to_datetime(hourly_pd["date"].astype(str))+ pd.to_timedelta(hourly_pd["hour"], unit="h"))
# hourly_pd["status_label"] = np.where(hourly_pd["iso_anomaly_flag"] == 1, "Anomaly", "Normal")

# print(hourly_pd["status_label"].value_counts(dropna=False))
# print(hourly_pd["status_label"].unique())

# @widgets.interact(call_type=sorted(hourly_pd['call_type'].unique()))
# def plot_by_call_type(call_type):
#     subset = hourly_pd[hourly_pd['call_type'] == call_type]
#     fig = px.box(subset, x='hour', y='avg_service_time', color='status_label',
#         title=f'{call_type} — Response Time by Hour (Isolation Forest Anomalies)',
#         labels={'hour': 'Hour of Day', 'avg_service_time': 'Avg Response Time (s)','status_label': 'Status'})
#     fig.update_layout(template='plotly_white', title_x=0.5)
#     fig.show()

The plot shows hourly 911 response times, with Isolation Forest marking unusually slow responses in red. Normal hours (blue) stay mostly within ~1,500–3,000 seconds, forming a consistent baseline. Anomalies appear as scattered red points with much longer delays—often 5,000–8,000 seconds—indicating isolated operational slowdowns rather than predictable daily patterns. The clear separation between red and blue distributions suggests the model is effectively flagging genuinely delayed response periods.

**Validating Call Type Mismatch Anomalies**

Most call type mismatches identified by the model represent minor wording or categorization differences rather than true labeling errors. This suggests the model is functioning correctly but that most detected mismatches are operationally routine, not anomalies of concern therfore there is no true need of validation for this type of anomaly detection.

**Validating Geographic Outlier Detection**

In [73]:
cols = ["dispatch_sector", "dispatch_latitude", "dispatch_longitude", "spatial_score", "is_geo_anomaly"]
geo_pd = (df_geo.select(*cols).sample(fraction=0.1, seed=42).toPandas()) #10 % of sample to make it grphable

print(f"Loaded {len(geo_pd):,} rows for validation sample")
geo_pd["status_label"] = geo_pd["is_geo_anomaly"].fillna(0).astype(int).map({0: "Normal", 1: "Anomaly"})

fig = px.box(
    geo_pd,
    x="status_label",
    y="spatial_score",
    color="status_label",
    title="Spatial Score Distribution (Normal vs Geographic Anomalies)",
    labels={"status_label": "Status", "spatial_score": "Spatial Score"},
)
#fig.update_layout(template="plotly_white", title_x=0.5, showlegend=False)
#fig.show()

Loaded 46,343 rows for validation sample


This box plot compares the spatial scores of normal calls versus geographic anomalies. Each point and box represents how far a call’s location deviates from where similar calls normally occur. The “Normal” group shows very low spatial scores clustered near zero, meaning these calls were located exactly where they are expected to be. The “Anomaly” group shows much higher spatial scores, indicating calls that were placed far outside typical patterns. A strong separation between the two boxes confirms that the spatial anomaly detector is effectively distinguishing expected locations from unusual or misplaced ones.

**Validating Temporal Burst Detection**

In [74]:
##########################
# Please uncomment to plot
#########################


# hourly_pd = df_w_cad_hourly.select("event_hour", "total_calls", "rolling_mean", "z_score", "is_burst").toPandas()

# hourly_pd = hourly_pd.sort_values("event_hour")
# hourly_pd["status_label"] = hourly_pd["is_burst"].fillna(0).astype(int).map({0: "Normal", 1: "Burst"})

# print("Burst status counts:")
# print(hourly_pd["status_label"].value_counts(dropna=False))
# print(f"Burst anomaly rate: {hourly_pd['is_burst'].mean():.2%}")

# fig = px.line(
#     hourly_pd,
#     x="event_hour",
#     y="total_calls",
#     color="status_label",
#     color_discrete_map={"Normal": "blue", "Burst": "red"},
#     title="Temporal Burst Detection — Citywide Call Volume",
#     labels={"event_hour": "Hour", "total_calls": "Total 911 Calls"},
# )
# fig.update_traces(mode="lines+markers")
# fig.update_layout(template="plotly_white", title_x=0.5)
# fig.show()

# burst_points = hourly_pd.query("is_burst == 1")
# fig2 = px.scatter(
#     burst_points,
#     x="event_hour",
#     y="total_calls",
#     color="z_score",
#     color_continuous_scale="Reds",
#     title="Detected Citywide Burst Events (Z-score Intensity)",
#     hover_data=["rolling_mean", "z_score"],
# )
# fig2.update_layout(template="plotly_white", title_x=0.5)
# fig2.show()

The top chart shows the total number of 911 calls each hour across the entire city, with colors distinguishing normal hours (blue) from burst hours (red). A burst hour is one where call volume spiked far above what the rolling window normally predicts, signaling a sudden, unusual surge in demand. When the red line rises above the blue cloud, it indicates a period where the city experienced an abnormally high number of calls.

The bottom chart isolates only those burst hours and shades each point by its z-score, where darker reds represent more extreme spikes. This helps you see not just when surges occurred, but also how intense they were relative to baseline conditions. Together, the two plots give both a full timeline of call behavior and a focused view of the most significant citywide bursts in 911 activity.

**Validating Dispatch Routing Anomolies**

In [75]:
##########################
# Please uncomment to plot
#########################


# dist_pd = df_dist.select("dispatch_sector", "distance_km", "z_score", "is_routing_anomaly").toPandas()

# dist_pd = dist_pd.dropna(subset=["distance_km"])
# dist_pd["status_label"] = dist_pd["is_routing_anomaly"].fillna(0).astype(int).map({0: "Normal", 1: "Anomaly"})

# print("Routing Anomaly Counts:")
# print(dist_pd["status_label"].value_counts(dropna=False))
# print(f"Routing anomaly rate: {dist_pd['is_routing_anomaly'].mean():.2%}")

# fig1 = px.box(
#     dist_pd,
#     x="status_label",
#     y="distance_km",
#     color="status_label",
#     color_discrete_map={"Normal": "blue", "Anomaly": "red"},
#     title="Dispatch Routing Anomalies — Distribution of Travel Distance",
#     labels={"status_label": "Status", "distance_km": "Travel Distance (km)"})
# fig1.update_layout(template="plotly_white", title_x=0.5)
# fig1.show()

# fig2 = px.box(
#     dist_pd,
#     x="dispatch_sector",
#     y="distance_km",
#     color="status_label",
#     color_discrete_map={"Normal": "blue", "Anomaly": "red"},
#     title="Dispatch Sector vs Travel Distance (Highlighting Routing Anomalies)",
#     labels={"dispatch_sector": "Dispatch Sector", "distance_km": "Travel Distance (km)"})
# fig2.update_layout(template="plotly_white", title_x=0.5, height=700)
# fig2.show()

# fig3 = px.scatter(
#     dist_pd,
#     x="z_score",
#     y="distance_km",
#     color="status_label",
#     color_discrete_map={"Normal": "blue", "Anomaly": "red"},
#     title="Z-Score vs Travel Distance (Routing Anomalies)",
#     labels={"z_score": "Distance Z-Score", "distance_km": "Travel Distance (km)"})
# fig3.update_layout(template="plotly_white", title_x=0.5)
# fig3.show()

The first chart compares how far units typically travel to reach a call versus how far they travel when a routing anomaly occurs. Normal dispatches (blue) stay clustered at short distances, reflecting units being assigned to calls within or near their sector. Anomalies (red), however, show much longer travel distances, indicating calls that were sent to a sector far from where the incident actually occurred.

The second chart breaks those distances down by dispatch sector, showing which specific sectors experienced unusually distant assignments. Most sectors have tightly grouped, short-distance calls in blue, while the red boxes reveal sectors where some calls were routed far outside their normal operating area.

The third chart plots each call’s travel distance against its distance-based z-score. Normal calls sit at low z-scores and short distances, whereas anomalies cluster in the upper-right with both very high z-scores and very long distances. This confirms that these outlier assignments are statistically extreme, not random variation. Together, the three charts show where routing was efficient, where it wasn’t, and how clearly the anomalies stand out from normal dispatch behavior.